# MVDOT Training — `train_main`

Warm-up pretraining followed by joint multi-view deep optimal transport (MVDOT) training on noisy MNIST. Restructured into stage-wise cells; logic is unchanged from the original script.

## Imports & Path Setup

Bring in the mvdot data, model, loss, and optimal-transport modules used below. *(path setup adjusted for notebook execution — see note below)*

In [8]:
import os
import sys

import torch
import torch.nn.functional as f

sys.path.append(
    os.path.dirname(
        os.getcwd()
    )
)

from mvdot.data.preprocessing import (
    load_mnist,
    create_noisy_view
)

from mvdot.data.dataset import noisy_mnist
from mvdot.data.dataloader import create_dataloader

from mvdot.models.autoencoder import autoencoder
from mvdot.models.matching import matching_network
from mvdot.models.barycenter import barycenter
from mvdot.models.transport_matching import (
    cross_view_affinity,
    build_bipartite_graph,
    bfs_topology,
    topology_cost,
    semantic_cost_from_probabilities,
    cross_view_transport
)

from mvdot.losses.reconstruction import reconstruction_loss
from mvdot.losses.manifold import (
    knn_graph,
    graph_laplacian,
    manifold_matching_loss
)

from mvdot.losses.semantic import (
    cluster_probabilities,
    total_semantic_matching_loss
)

from mvdot.ot.barycenter_ot import (
    initialize_barycenter
)

from mvdot.ot.transport import (
    sample_to_cluster_transport
)


def normalized_manifold_matching_loss(
    transport,
    cost,
    z,
    graph,
    lambda_graph
):
    transport_cost = torch.sum(transport * cost)
    indices = graph.indices()
    values = graph.values()
    pairwise_distance = (
        z[indices[0]] - z[indices[1]]
    ).pow(2).sum(dim=1)
    graph_cost = 0.5 * torch.sum(
        values * pairwise_distance
    ) / values.sum().clamp_min(1.0)
    return (
        transport_cost + lambda_graph * graph_cost,
        transport_cost,
        graph_cost
    )


def paper_semantic_matching_loss(
    transport,
    probabilities1,
    probabilities2
):
    compatibility = torch.clamp(
        probabilities1 @ probabilities2.t(),
        min=1e-12
    )

    return -torch.sum(
        transport * torch.log(compatibility)
    )


def paper_total_semantic_matching_loss(
    transports,
    probabilities,
    transport_costs
):
    distances = torch.stack(transport_costs)
    inverse_distances = 1.0 / distances.clamp_min(1e-8)
    weights = inverse_distances / inverse_distances.sum().clamp_min(1e-8)

    loss = torch.zeros(
        (),
        device=probabilities[0].device,
        dtype=probabilities[0].dtype
    )

    for m in range(len(probabilities)):
        for n in range(len(probabilities)):
            if m == n:
                continue

            loss = loss + (
                weights[m] * weights[n]
                * paper_semantic_matching_loss(
                    transports[(m, n)],
                    probabilities[m],
                    probabilities[n]
                )
            )

    return loss, weights


## Configuration & Reproducibility

Device selection, all hyperparameters, and the random seed for the run.

In [9]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

input_dim = 784
hidden_dim = 512
latent_dim = 128
num_clusters = 10

batch_size = 256

warmup_epochs = 10
train_epochs = 50

aligned_rate = 0.5
sigma = 0.25

learning_rate = 3e-4

epsilon = 0.1

lambda_graph = 0.05

alpha = 0.98

knn_k = 10
cross_view_top_k = 5

sinkhorn_iterations = 100

seed = 42


torch.manual_seed(
    seed
)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(
        seed
    )


<!-- ## Load & Preprocess Data

Load raw MNIST images/labels, build the noisy second view, and wrap both in a DataLoader. -->

In [10]:
print(
    "device:",
    device
)

print(
    "loading data..."
)


images, labels = load_mnist(
    "data"
)

noisy_images = create_noisy_view(
    images,
    sigma=sigma,
    seed=seed
)


dataset = noisy_mnist(
    images,
    noisy_images,
    labels,
    aligned_rate=aligned_rate,
    seed=seed
)


loader = torch.utils.data.DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=True,
    drop_last=False
)


print(
    "samples:",
    len(dataset)
)

print(
    "batches:",
    len(loader)
)

print(
    "aligned rate:",
    aligned_rate
)

print(
    "noise sigma:",
    sigma
)


device: cpu
loading data...
samples: 60000
batches: 234
aligned rate: 0.5
noise sigma: 0.25


## Initialize Models, Loss & Warm-up Optimizer

Create the two per-view autoencoders, the matching network, the barycenter module, and the reconstruction loss.

In [11]:
model1 = autoencoder(
    input_dim=input_dim,
    hidden_dim=hidden_dim,
    latent_dim=latent_dim
).to(device)

model2 = autoencoder(
    input_dim=input_dim,
    hidden_dim=hidden_dim,
    latent_dim=latent_dim
).to(device)

matching = matching_network(
    feature_dim=latent_dim
).to(device)

bary = barycenter(
    num_clusters=num_clusters,
    feature_dim=latent_dim
).to(device)


criterion = reconstruction_loss()


warmup_optimizer = torch.optim.Adam(
    list(model1.parameters())
    + list(model2.parameters()),
    lr=learning_rate
)


## Stage 1 — Autoencoder Warm-up Training

Pretrain the two autoencoders on reconstruction loss alone before the joint objective is introduced.

In [12]:
print()
print(
    "starting autoencoder warm-up..."
)


model1.train()
model2.train()

for epoch in range(
    warmup_epochs
):
    epoch_loss = 0.0

    for view1, view2, _ in loader:

        view1 = view1.to(
            device,
            non_blocking=True
        )

        view2 = view2.to(
            device,
            non_blocking=True
        )

        warmup_optimizer.zero_grad(
            set_to_none=True
        )

        z1, xhat1 = model1(
            view1
        )

        z2, xhat2 = model2(
            view2
        )

        loss1 = criterion(
            view1,
            xhat1
        )

        loss2 = criterion(
            view2,
            xhat2
        )

        loss = (
            loss1
            + loss2
        )

        loss.backward()

        warmup_optimizer.step()

        epoch_loss += loss.item()

    epoch_loss /= len(
        loader
    )

    print(
        "warmup epoch:",
        epoch + 1,
        "/",
        warmup_epochs,
        "loss:",
        epoch_loss
    )


print()
print(
    "autoencoder warm-up complete"
)



starting autoencoder warm-up...
warmup epoch: 1 / 10 loss: 0.13227326601234257
warmup epoch: 2 / 10 loss: 0.06955143172516783
warmup epoch: 3 / 10 loss: 0.054124808018533595
warmup epoch: 4 / 10 loss: 0.04677162265293618
warmup epoch: 5 / 10 loss: 0.04248664911804546
warmup epoch: 6 / 10 loss: 0.03962212078209616
warmup epoch: 7 / 10 loss: 0.0374029414393963
warmup epoch: 8 / 10 loss: 0.035696835949634895
warmup epoch: 9 / 10 loss: 0.034353726217125215
warmup epoch: 10 / 10 loss: 0.033169918940362766

autoencoder warm-up complete


## Extract Warm-up Representations & Initialize Barycenter

Encode the full dataset with the warmed-up models and use the resulting features to initialize the barycenter's cluster centers and weights.

In [13]:
print(
    "extracting warm-up representations..."
)


model1.eval()
model2.eval()
matching.eval()


all_h1 = []
all_h2 = []


with torch.no_grad():

    for view1, view2, _ in loader:

        view1 = view1.to(
            device,
            non_blocking=True
        )

        view2 = view2.to(
            device,
            non_blocking=True
        )

        z1 = model1.encode(
            view1
        )

        z2 = model2.encode(
            view2
        )

        h1 = matching(
            z1
        )

        h2 = matching(
            z2
        )

        all_h1.append(
            h1
        )

        all_h2.append(
            h2
        )


h1_all = torch.cat(
    all_h1,
    dim=0
)

h2_all = torch.cat(
    all_h2,
    dim=0
)


initial_features = torch.cat(
    [
        h1_all,
        h2_all
    ],
    dim=0
)


print(
    "initializing barycenter..."
)


initial_centers, initial_weights = (
    initialize_barycenter(
        initial_features,
        num_clusters=num_clusters,
        iterations=20,
        seed=seed
    )
)


bary.set_centers(
    initial_centers
)

bary.weights.copy_(
    initial_weights
)


del h1_all
del h2_all
del initial_features
del all_h1
del all_h2


if torch.cuda.is_available():
    torch.cuda.empty_cache()


print(
    "centers shape:",
    bary.centers.shape
)

print(
    "weights:",
    bary.weights
)


extracting warm-up representations...
initializing barycenter...
centers shape: torch.Size([10, 128])
weights: tensor([0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000,
        0.1000])


## Stage 2 — Joint MVDOT Training

Full joint training: reconstruction + manifold-matching + semantic-matching losses, with the barycenter updated each step.

In [14]:
optimizer = torch.optim.Adam(
    list(model1.parameters())
    + list(model2.parameters())
    + list(matching.parameters())
    + list(bary.parameters()),
    lr=learning_rate
)


print()
print(
    "starting mvdot training..."
)


model1.train()
model2.train()
matching.train()
bary.train()


for epoch in range(
    train_epochs
):

    epoch_loss = 0.0
    epoch_lrec = 0.0
    epoch_lmm = 0.0
    epoch_lsm = 0.0

    for batch_index, (
        view1,
        view2,
        _
    ) in enumerate(loader):

        view1 = view1.to(
            device,
            non_blocking=True
        )

        view2 = view2.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        z1, xhat1 = model1(
            view1
        )

        z2, xhat2 = model2(
            view2
        )

        h1 = matching(
            z1
        )

        h2 = matching(
            z2
        )

        centers, weights = bary()

        lrec1 = criterion(
            view1,
            xhat1
        )

        lrec2 = criterion(
            view2,
            xhat2
        )

        lrec = (
            lrec1
            + lrec2
        )

        t1, cost1 = (
            sample_to_cluster_transport(
                h1,
                centers,
                weights,
                epsilon=epsilon,
                iterations=sinkhorn_iterations
            )
        )

        t2, cost2 = (
            sample_to_cluster_transport(
                h2,
                centers,
                weights,
                epsilon=epsilon,
                iterations=sinkhorn_iterations
            )
        )

        graph1 = knn_graph(
            z1,
            k=knn_k,
            chunk_size=64
        )

        graph2 = knn_graph(
            z2,
            k=knn_k,
            chunk_size=64
        )

        degree1 = graph_laplacian(
            graph1
        )

        degree2 = graph_laplacian(
            graph2
        )

        manifold1 = normalized_manifold_matching_loss(
            t1,
            cost1,
            z1,
            graph1,
            degree1,
            lambda_graph=lambda_graph
        )

        manifold2 = normalized_manifold_matching_loss(
            t2,
            cost2,
            z2,
            graph2,
            degree2,
            lambda_graph=lambda_graph
        )

        transport_cost1 = manifold1[1]
        transport_cost2 = manifold2[1]

        graph_cost1 = manifold1[2]
        graph_cost2 = manifold2[2]

        lmm = (
            manifold1[0]
            + manifold2[0]
        )

        probabilities1 = cluster_probabilities(
            h1,
            centers
        )

        probabilities2 = cluster_probabilities(
            h2,
            centers
        )

        edges, _ = cross_view_affinity(
            h1,
            h2,
            centers,
            sigma=1.0,
            top_k=cross_view_top_k
        )

        graph_edges = build_bipartite_graph(
            edges,
            h1.size(0),
            h2.size(0)
        )

        topology = bfs_topology(
            graph_edges,
            h1.size(0),
            h2.size(0)
        ).to(
            device
        )

        topology_matrix = topology_cost(
            topology
        )

        semantic_matrix = (
            semantic_cost_from_probabilities(
                probabilities1,
                probabilities2
            )
        )

        p12, _ = cross_view_transport(
            semantic_matrix,
            topology_matrix,
            epsilon=epsilon,
            iterations=sinkhorn_iterations
        )

        p21 = p12.t()

        transports = {
            (0, 1): p12,
            (1, 0): p21
        }

        probabilities = [
            probabilities1,
            probabilities2
        ]

        transport_costs = [
            transport_cost1.detach(),
            transport_cost2.detach()
        ]

        lsm, view_weights = (
            paper_total_semantic_matching_loss(
                transports,
                probabilities,
                transport_costs
            )
        )

        loss = (
            lrec
            + lmm
            + lsm
        )

        if not torch.isfinite(
            loss
        ):
            print(
                "non-finite loss at batch:",
                batch_index + 1
            )
            raise RuntimeError(
                "non-finite loss encountered"
            )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            list(model1.parameters())
            + list(model2.parameters())
            + list(matching.parameters())
            + list(bary.parameters()),
            max_norm=5.0
        )

        optimizer.step()

        with torch.no_grad():
            bary.normalize_centers()
            bary.update_weights(
                [
                    t1.detach(),
                    t2.detach()
                ],
                momentum=alpha
            )

        epoch_loss += loss.item()
        epoch_lrec += lrec.item()
        epoch_lmm += lmm.item()
        epoch_lsm += lsm.item()

        if (
            batch_index == 0
            or (batch_index + 1) % 25 == 0
            or batch_index + 1 == len(loader)
        ):
            print(
                "epoch:",
                epoch + 1,
                "/",
                train_epochs,
                "batch:",
                batch_index + 1,
                "/",
                len(loader),
                "loss:",
                loss.item()
            )

    epoch_loss /= len(loader)
    epoch_lrec /= len(loader)
    epoch_lmm /= len(loader)
    epoch_lsm /= len(loader)

    print()
    print(
        "epoch:",
        epoch + 1,
        "average loss:",
        epoch_loss
    )
    print(
        "lrec:",
        epoch_lrec
    )
    print(
        "lmm:",
        epoch_lmm
    )
    print(
        "lsm:",
        epoch_lsm
    )
    print(
        "view weights:",
        view_weights.detach()
    )
    print(
        "barycenter weight sum:",
        bary.weights.sum().item()
    )
    print()



starting mvdot training...
epoch: 1 / 10 batch: 1 / 234 loss: 57644.3046875
epoch: 1 / 10 batch: 25 / 234 loss: 27745.5546875
epoch: 1 / 10 batch: 50 / 234 loss: 12653.39453125
epoch: 1 / 10 batch: 75 / 234 loss: 7247.3291015625
epoch: 1 / 10 batch: 100 / 234 loss: 4235.62744140625
epoch: 1 / 10 batch: 125 / 234 loss: 2619.6728515625


epoch: 1 / 10 batch: 150 / 234 loss: 1861.43310546875
epoch: 1 / 10 batch: 175 / 234 loss: 1257.7032470703125
epoch: 1 / 10 batch: 200 / 234 loss: 983.8200073242188
epoch: 1 / 10 batch: 225 / 234 loss: 719.999267578125
epoch: 1 / 10 batch: 234 / 234 loss: 667.8184814453125

epoch: 1 average loss: 9017.400179088625
lrec: 0.08027914852604397
lmm: 9018.470942831447
lsm: -1.1510482292908888
view weights: tensor([0.4917, 0.5083])
barycenter weight sum: 1.0000001192092896

epoch: 2 / 10 batch: 1 / 234 loss: 624.3095703125
epoch: 2 / 10 batch: 25 / 234 loss: 477.81787109375
epoch: 2 / 10 batch: 50 / 234 loss: 405.5969543457031
epoch: 2 / 10 batch: 75 / 234 loss: 323.21588134765625
epoch: 2 / 10 batch: 100 / 234 loss: 272.6222229003906
epoch: 2 / 10 batch: 125 / 234 loss: 213.93006896972656
epoch: 2 / 10 batch: 150 / 234 loss: 191.0781707763672
epoch: 2 / 10 batch: 175 / 234 loss: 159.7303466796875
epoch: 2 / 10 batch: 200 / 234 loss: 138.91677856445312
epoch: 2 / 10 batch: 225 / 234 loss: 119

## Save Final Checkpoint

Persist model weights and every hyperparameter needed to reconstruct this run (used later by evaluate/inference).

In [15]:
os.makedirs(
    "checkpoints",
    exist_ok=True
)


checkpoint = {
    "model1": model1.state_dict(),
    "model2": model2.state_dict(),
    "matching": matching.state_dict(),
    "barycenter": bary.state_dict(),
    "input_dim": input_dim,
    "hidden_dim": hidden_dim,
    "latent_dim": latent_dim,
    "num_clusters": num_clusters,
    "aligned_rate": aligned_rate,
    "sigma": sigma,
    "batch_size": batch_size,
    "learning_rate": learning_rate,
    "epsilon": epsilon,
    "lambda_graph": lambda_graph,
    "alpha": alpha,
    "knn_k": knn_k,
    "cross_view_top_k": cross_view_top_k,
    "sinkhorn_iterations": sinkhorn_iterations,
    "warmup_epochs": warmup_epochs,
    "train_epochs": train_epochs,
    "seed": seed
}


checkpoint_path = (
    "checkpoints/mvdot_final.pt"
)


torch.save(
    checkpoint,
    checkpoint_path
)


print()
print(
    "checkpoint saved"
)

print(
    "path:",
    checkpoint_path
)

print()
print(
    "mvdot training complete"
)




checkpoint saved
path: checkpoints/mvdot_final.pt

mvdot training complete
